# RoadSense — Day 1: Train the road-damage detector

**What this notebook does, step by step:**
1. Connects to a free GPU and your Google Drive.
2. Downloads real road photos (RDD2022 dataset, India + Japan).
3. Converts the labels into the format YOLO understands.
4. Shows you some photos with their boxes, so you can check the data is correct.
5. Runs a 2-minute **smoke test** (a tiny training run to make sure nothing is broken).
6. Runs the **real training** (a few hours, saved to Drive so a disconnect doesn't lose work).
7. Measures accuracy (**mAP**) and saves the numbers for your README and resume.
8. Exports the model to **ONNX** (a light file format our backend server can run).

### Before you start
- Menu **Runtime → Change runtime type → T4 GPU → Save**.
- Run cells **one at a time, top to bottom** (click a cell, press **Shift + Enter**).
- Each cell has a comment at the top explaining what it does. Read it before running.

## Step 1 — Check the GPU and connect Google Drive

In [ ]:
# A GPU (graphics card) makes training ~20-50x faster than a normal CPU.
# This command prints the GPU name. You should see "Tesla T4".
# If you see an error instead, go to Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi

In [ ]:
# Google Drive is where we save training progress and the final model.
# Colab deletes everything else when the session ends, Drive keeps it.
# A pop-up will ask you to allow access: click through and allow.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/roadsense'   # your project folder in Drive
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Saving everything important to:', DRIVE_DIR)

## Step 2 — Install the libraries

In [ ]:
# ultralytics  = the library that contains YOLO and does the training for us
# onnx, onnxruntime = to export the model to ONNX and test that the exported file works
!pip install -q -U ultralytics onnx onnxruntime
import ultralytics
ultralytics.checks()   # prints versions + confirms the GPU is visible

## Step 3 — Get the dataset

The original per-country download links no longer work, so we use the dataset's
**official backup copy on figshare**. It's one big 13.3 GB file with all six countries.

This cell:
1. downloads that big file (**10–30 minutes**, depending on speed),
2. checks the download is complete (it compares the exact file size),
3. pulls out **only India and Japan** (we don't need the other 4 countries),
4. deletes the big file to free up space,
5. saves a **small copy (India + Japan only) to your Google Drive**.

Because of step 5, if Colab ever disconnects, running this cell again restores the
data from Drive in about 2 minutes instead of downloading 13 GB again.

**If the download stops halfway**, just run the cell again. It continues from
where it stopped.

In [ ]:
import os, zipfile, shutil

RAW = '/content/raw'                                  # where the India + Japan photos will live
COUNTRIES = ['India', 'Japan']
CACHE = f'{DRIVE_DIR}/rdd_india_japan.zip'           # the small copy saved in your Drive
BIG = '/content/RDD2022_full.zip'                     # the big 13.3 GB download (deleted after use)
BIG_URL = 'https://ndownloader.figshare.com/files/38030910'   # official figshare mirror
BIG_SIZE = 13264172619                                # exact size of the complete file, in bytes

def extract_countries(big_zip, raw_dir, countries):
    # Pull only the chosen countries out of the big zip.
    # Works whether the big zip holds folders (India/...) or one zip file per country.
    with zipfile.ZipFile(big_zip) as z:
        names = z.namelist()
        print('Files inside the big zip:', len(names))
        for c in countries:
            target = f'{raw_dir}/{c}'
            os.makedirs(target, exist_ok=True)
            inner = [n for n in names if n.lower().endswith('.zip')
                     and c.lower() in os.path.basename(n).lower()]
            members = [n for n in names if f'/{c}/' in '/' + n]
            if inner:                                 # case A: a zip inside the zip
                for n in inner:
                    print(f'  {c}: unpacking inner zip {n}')
                    tmp = z.extract(n, '/content/tmp_inner')
                    with zipfile.ZipFile(tmp) as iz:
                        iz.extractall(target)
                    os.remove(tmp)
            elif members:                             # case B: plain folders
                print(f'  {c}: extracting {len(members)} files')
                for n in members:
                    z.extract(n, target)
            else:
                raise RuntimeError(f'Could not find {c} in the zip. Send Claude this list: {names[:40]}')

def have_data():
    return all(os.path.isdir(f'{RAW}/{c}') and os.listdir(f'{RAW}/{c}') for c in COUNTRIES)

os.makedirs(RAW, exist_ok=True)
if have_data():
    print('Data is already here, nothing to do.')
elif os.path.exists(CACHE):
    print('Restoring India + Japan from your Drive copy (about 2 minutes) ...')
    !unzip -q -o "{CACHE}" -d {RAW}
else:
    have = os.path.getsize(BIG) if os.path.exists(BIG) else 0
    if have != BIG_SIZE:
        print('Downloading the full RDD2022 dataset (13.3 GB). This can take 10-30 minutes ...')
        !wget -c -q --show-progress -O {BIG} "{BIG_URL}"
    have = os.path.getsize(BIG) if os.path.exists(BIG) else 0
    if have != BIG_SIZE:
        raise RuntimeError(f'Download incomplete: {have:,} of {BIG_SIZE:,} bytes. '
                           'Run this cell again, it will continue where it stopped.')
    print('Download complete and verified. Extracting India + Japan ...')
    extract_countries(BIG, RAW, COUNTRIES)
    os.remove(BIG)                                    # free 13 GB of disk
    print('Saving a small copy to your Drive for next time (a few minutes) ...')
    !cd {RAW} && zip -q -r -0 "{CACHE}" India Japan

import glob
for c in COUNTRIES:
    n = len(glob.glob(f'{RAW}/{c}/**/train/**/*.xml', recursive=True))
    print(f'{c}: {n} annotation files found')

## Step 4 — Look at what we downloaded

Each training image has a matching **.xml** file that lists the damage boxes.
This cell finds all those files and counts how many boxes of each damage type exist.

The damage codes mean:
| Code | Meaning |
|---|---|
| D00 | Longitudinal crack (runs along the road) |
| D10 | Transverse crack (runs across the road) |
| D20 | Alligator crack (looks like crocodile skin) |
| D40 | Pothole |

You may see other codes too (like D43, D44, D50). Those are rarer extra
categories. We ignore them and keep only the four main ones.

In [ ]:
import glob, collections
import xml.etree.ElementTree as ET

# Find every annotation file, no matter how the zip arranged its folders.
xml_files = sorted(glob.glob(f'{RAW}/**/train/**/*.xml', recursive=True))
print('Annotation files found:', len(xml_files))

counts = collections.Counter()
for xf in xml_files:
    for obj in ET.parse(xf).getroot().iter('object'):
        counts[obj.findtext('name', '').strip()] += 1

print('\nBoxes per damage code:')
for name, n in counts.most_common():
    print(f'  {name:>6}: {n}')

## Step 5 — Convert labels to YOLO format

**Why convert?** The dataset stores boxes as corner pixels:
`xmin, ymin, xmax, ymax` (Pascal VOC format, in XML).

YOLO wants one `.txt` file per image, one line per box:
```
class_id  x_center  y_center  width  height
```
where every number is a **fraction of the image size** (between 0 and 1).

**Worked example.** An image 600 px wide and 600 px tall has a pothole
with corners xmin=150, ymin=300, xmax=270, ymax=420.
- x_center = (150+270)/2 / 600 = 210/600 = **0.35**
- y_center = (300+420)/2 / 600 = 360/600 = **0.60**
- width    = (270-150)/600 = 120/600 = **0.20**
- height   = (420-300)/600 = 120/600 = **0.20**
- pothole is class **3**, so the line is: `3 0.350000 0.600000 0.200000 0.200000`

We also split the images: **85% for training** (the model learns from these) and
**15% for validation** (held back, used only to measure accuracy honestly).

In [ ]:
import os, glob, random, shutil
import xml.etree.ElementTree as ET
from PIL import Image

CLASS_MAP = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}
CLASS_NAMES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']

def find_image_for_xml(xml_path):
    # RDD2022 layout: .../train/annotations/xmls/X.xml  ->  .../train/images/X.jpg
    stem = os.path.splitext(os.path.basename(xml_path))[0]
    train_dir = xml_path.split('/annotations/')[0]
    for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
        p = os.path.join(train_dir, 'images', stem + ext)
        if os.path.exists(p):
            return p
    return None

def voc_to_yolo_lines(xml_path, img_path):
    # Returns the YOLO label lines for one image (an empty list = no damage, still useful).
    root = ET.parse(xml_path).getroot()
    try:
        W = int(float(root.findtext('size/width')))
        H = int(float(root.findtext('size/height')))
    except (TypeError, ValueError):
        W = H = 0
    if W <= 0 or H <= 0:                     # some files have a missing size: read the image
        with Image.open(img_path) as im:
            W, H = im.size
    lines = []
    for obj in root.iter('object'):
        name = obj.findtext('name', '').strip()
        if name not in CLASS_MAP:
            continue                          # ignore the rare extra categories
        bb = obj.find('bndbox')
        x1, y1 = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
        x2, y2 = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
        # keep the box inside the image, then skip broken (zero-size) boxes
        x1, x2 = max(0, min(x1, W)), max(0, min(x2, W))
        y1, y2 = max(0, min(y1, H)), max(0, min(y2, H))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        xc, yc = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        lines.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

def build_yolo_dataset(raw_dir, out_dir, val_fraction=0.15, seed=42):
    random.seed(seed)
    for split in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{split}', exist_ok=True)
    xmls = sorted(glob.glob(f'{raw_dir}/**/train/**/*.xml', recursive=True))
    # group by country so each country is split 85/15 separately (fair validation set)
    by_country = {}
    for x in xmls:
        country = os.path.relpath(x, raw_dir).split('/')[0]
        by_country.setdefault(country, []).append(x)
    stats = {'train': 0, 'val': 0, 'missing_image': 0, 'boxes': 0}
    for country, files in by_country.items():
        random.shuffle(files)
        n_val = int(len(files) * val_fraction)
        for i, xml_path in enumerate(files):
            split = 'val' if i < n_val else 'train'
            img = find_image_for_xml(xml_path)
            if img is None:
                stats['missing_image'] += 1
                continue
            lines = voc_to_yolo_lines(xml_path, img)
            stem = f"{country}_{os.path.splitext(os.path.basename(img))[0]}"   # unique name
            ext = os.path.splitext(img)[1].lower()
            shutil.copy(img, f'{out_dir}/images/{split}/{stem}{ext}')
            with open(f'{out_dir}/labels/{split}/{stem}.txt', 'w') as f:
                f.write('\n'.join(lines))
            stats[split] += 1
            stats['boxes'] += len(lines)
    return stats

DATA = '/content/rdd_yolo'
if os.path.exists(DATA):
    shutil.rmtree(DATA)                        # start fresh if you re-run this cell
stats = build_yolo_dataset(RAW, DATA)
print(stats)

In [ ]:
# data.yaml tells YOLO where the images are and what the class names are.
import yaml
data_yaml = {
    'path': DATA,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
with open(f'{DATA}/data.yaml', 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(open(f'{DATA}/data.yaml').read())

## Step 6 — Check the labels with your own eyes

Never skip this. If the conversion had a bug, the boxes would land in the wrong
places, and the model would learn garbage. Look at the pictures: do the boxes sit
on cracks and potholes?

In [ ]:
import random, glob, os
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COLORS = ['yellow', 'cyan', 'orange', 'red']   # one colour per damage type
imgs = [p for p in glob.glob(f'{DATA}/images/train/*')
        if os.path.getsize(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt') > 0]
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, p in zip(axes.flat, random.sample(imgs, 6)):
    im = Image.open(p).convert('RGB'); W, H = im.size; d = ImageDraw.Draw(im)
    for line in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
        c, xc, yc, w, h = map(float, line.split())
        x1, y1 = (xc - w/2) * W, (yc - h/2) * H
        x2, y2 = (xc + w/2) * W, (yc + h/2) * H
        d.rectangle([x1, y1, x2, y2], outline=COLORS[int(c)], width=4)
        d.text((x1 + 4, y1 + 4), CLASS_NAMES[int(c)], fill=COLORS[int(c)])
    ax.imshow(im); ax.set_title(os.path.basename(p)); ax.axis('off')
plt.tight_layout(); plt.show()

## Step 7 — Smoke test (about 2–3 minutes)

A tiny training run on 10% of the data for 1 epoch. We are **not** trying to get a
good model here. We only check that everything runs without errors before
spending hours on the real run.

In [ ]:
from ultralytics import YOLO
smoke = YOLO('yolo26n.pt')          # 'n' = nano: the smallest, fastest YOLO26 model
smoke.train(data=f'{DATA}/data.yaml', epochs=1, imgsz=640, batch=16,
            fraction=0.1, project='/content/smoke', name='test', exist_ok=True, plots=False)
print('\nSmoke test finished without errors. You can start the real training.')

## Step 8 — Real training (a few hours)

What the settings mean:
- `epochs=60`: go through all training images up to 60 times.
- `patience=15`: stop early if accuracy hasn't improved for 15 epochs (saves time).
- `imgsz=640`: resize photos to 640×640 pixels.
- `project=DRIVE_DIR...`: save progress to Google Drive after every epoch.

You'll see a table updating each epoch. The column to watch is **mAP50**: it should
generally go up. **Leave this tab open.** You can work on Day 2 setup meanwhile.

In [ ]:
from ultralytics import YOLO
RUN_DIR = f'{DRIVE_DIR}/runs'
model = YOLO('yolo26n.pt')
model.train(data=f'{DATA}/data.yaml', epochs=60, patience=15, imgsz=640, batch=16,
            project=RUN_DIR, name='rdd_y26n', exist_ok=True, seed=42)

### Step 8b — ONLY if Colab disconnected during training

When you come back after a disconnect:
1. Runtime → Change runtime type → T4 GPU (if needed).
2. Run **Step 1, Step 2, Step 3 and Step 5** again. This is needed because the
   dataset lived on the temporary disk and was wiped. Step 3 restores it from your
   Drive copy in about 2 minutes. Skip Step 4, Step 6 and Step 7.
3. Then run the cell below. It continues from the last saved epoch in Drive.

In [ ]:
from ultralytics import YOLO
last = f'{DRIVE_DIR}/runs/rdd_y26n/weights/last.pt'
YOLO(last).train(resume=True)

## Step 9 — Measure accuracy on the validation set

**mAP50** is the main score: it counts a prediction as correct if its box overlaps
the real box by at least 50%. **mAP50-95** is stricter (it averages over overlaps
from 50% to 95%). The per-class numbers tell you which damage types the model is good
and bad at. Write these down, they go in your README and resume.

In [ ]:
import json
from ultralytics import YOLO
BEST = f'{DRIVE_DIR}/runs/rdd_y26n/weights/best.pt'
best = YOLO(BEST)
m = best.val(data=f'{DATA}/data.yaml', imgsz=640, split='val')

# ap_class_index tells us which class each score belongs to
per_class = {CLASS_NAMES[int(c)]: round(float(m.box.ap50[i]), 4)
             for i, c in enumerate(m.box.ap_class_index)}
results = {
    'model': 'YOLO26n',
    'train_images': stats['train'], 'val_images': stats['val'],
    'mAP50': round(float(m.box.map50), 4),
    'mAP50_95': round(float(m.box.map), 4),
    'precision': round(float(m.box.mp), 4),
    'recall': round(float(m.box.mr), 4),
    'AP50_per_class': per_class,
}
print(json.dumps(results, indent=2))
with open(f'{DRIVE_DIR}/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved to', f'{DRIVE_DIR}/metrics.json')

## Step 10 — See the model's predictions on photos it has never seen

In [ ]:
import glob, random
import matplotlib.pyplot as plt
val_imgs = random.sample(glob.glob(f'{DATA}/images/val/*'), 6)
preds = best.predict(val_imgs, imgsz=640, conf=0.25, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, r in zip(axes.flat, preds):
    ax.imshow(r.plot()[:, :, ::-1])   # r.plot() draws boxes; [::-1] converts BGR -> RGB colours
    ax.set_title(f'{len(r.boxes)} detections'); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/sample_predictions.png', dpi=80); plt.show()

## Step 11 — Export to ONNX and test the exported file

Our free server has only 512 MB of memory, and PyTorch alone would nearly fill it.
ONNX lets the server run the model with the much lighter `onnxruntime` library.

The last line prints the **output shape**. Please copy it and send it to me,
along with the metrics from Step 9. I'll write the backend code to match it exactly.

In [ ]:
import shutil, os, numpy as np, onnxruntime as ort
onnx_path = best.export(format='onnx', imgsz=640)          # creates best.onnx next to best.pt
final = f'{DRIVE_DIR}/roadsense.onnx'
shutil.copy(onnx_path, final)

sess = ort.InferenceSession(final, providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.run(None, {inp.name: np.zeros((1, 3, 640, 640), dtype=np.float32)})
print('Input :', inp.name, inp.shape)
print('Output shape:', [o.shape for o in out])
print(f'File size: {os.path.getsize(final) / 1e6:.1f} MB  ->  saved at {final}')

## Done with Day 1

In your Google Drive folder **roadsense/** you should now have:
- `roadsense.onnx`: the trained model (goes into the backend on Day 2)
- `metrics.json`: your accuracy numbers (for the README and resume)
- `sample_predictions.png`: a picture for your README
- `runs/rdd_y26n/`: training charts (`results.png`, `confusion_matrix.png`), which are nice for the README

**Send me:** the metrics printed in Step 9 and the "Output shape" line from Step 11.